# BDA 수강 완료 예측 파이프라인
Target: `completed` (0/1 이진 분류)

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

DATA_PATH = '/content/drive/MyDrive/open'

# 한글 폰트 설정 (Colab)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fe = fm.FontEntry(fname=font_path, name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

DATA_PATH = 'content/drive/MyDrive/open'

plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

## 1. 데이터 로드

In [ ]:
train = pd.read_csv(f'{DATA_PATH}/train.csv')
test  = pd.read_csv(f'{DATA_PATH}/test.csv')
sub   = pd.read_csv(f'{DATA_PATH}/sample_submission.csv')

print(f'train shape : {train.shape}')
print(f'test  shape : {test.shape}')
train.head(3)

## 2. EDA

In [ ]:
# 기본 정보
print('=== 데이터 타입 & 결측치 ===' )
info_df = pd.DataFrame({
    'dtype'   : train.dtypes,
    'null_cnt': train.isnull().sum(),
    'null_pct': (train.isnull().mean() * 100).round(2),
    'nunique' : train.nunique()
})
display(info_df[info_df['null_cnt'] > 0])
print(f'\n총 결측 컬럼 수: {(train.isnull().sum() > 0).sum()}')

In [ ]:
# 타겟 분포
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vc = train['completed'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values, color=['#5B9BD5','#ED7D31'])
axes[0].set_title('Target 분포 (completed)')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=12)

axes[1].pie(vc.values, labels=['미완료(0)', '완료(1)'], autopct='%1.1f%%',
            colors=['#5B9BD5','#ED7D31'], startangle=90)
axes[1].set_title('Target 비율')

plt.tight_layout()
plt.show()
print(vc)

In [ ]:
# 수치형 피처 분포
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['ID', 'completed']]
print(f'수치형 컬럼: {num_cols}')

if num_cols:
    fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 4 * len(num_cols)))
    if len(num_cols) == 1:
        axes = [axes]
    for i, col in enumerate(num_cols):
        train[col].hist(ax=axes[i][0], bins=30, color='steelblue', edgecolor='white')
        axes[i][0].set_title(f'{col} 분포')
        train.boxplot(column=col, by='completed', ax=axes[i][1])
        axes[i][1].set_title(f'{col} by completed')
    plt.tight_layout()
    plt.show()

In [ ]:
# 카테고리형 피처 top-10 빈도
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'ID']
print(f'카테고리형 컬럼 수: {len(cat_cols)}')

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(14, 4 * len(cat_cols)))
if len(cat_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_cols):
    vc = train[col].value_counts().head(10)
    ax.barh(vc.index.astype(str), vc.values, color='steelblue')
    ax.set_title(f'{col} (top 10)')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 상관관계
if num_cols:
    corr = train[num_cols + ['completed']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
    plt.title('상관관계 히트맵')
    plt.tight_layout()
    plt.show()

## 3. 전처리

In [ ]:
def preprocess(train_df, test_df):
    """train/test 동일하게 전처리 후 반환"""
    drop_cols = ['ID']
    target_col = 'completed'

    y = train_df[target_col].copy()

    tr = train_df.drop(columns=drop_cols + [target_col], errors='ignore').copy()
    te = test_df.drop(columns=drop_cols + [target_col], errors='ignore').copy()

    all_data = pd.concat([tr, te], axis=0, ignore_index=True)

    # bool dtype → int
    for col in all_data.select_dtypes(include='bool').columns:
        all_data[col] = all_data[col].astype(int)

    # 'True'/'False' 문자열 → int
    for col in all_data.select_dtypes(include='object').columns:
        unique_vals = set(all_data[col].dropna().unique())
        if unique_vals <= {'True', 'False'}:
            all_data[col] = all_data[col].map({'True': 1, 'False': 0})

    # 카테고리형 → Label Encoding
    cat_cols = all_data.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        all_data[col] = le.fit_transform(all_data[col].astype(str))

    # 결측치 처리: 전체 NaN 컬럼도 유지 (keep_empty_features=True)
    num_cols = all_data.select_dtypes(include='number').columns.tolist()
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    all_data[num_cols] = imputer.fit_transform(all_data[num_cols])

    X_train = all_data.iloc[:len(tr)].reset_index(drop=True)
    X_test  = all_data.iloc[len(tr):].reset_index(drop=True)

    return X_train, X_test, y, cat_cols

X, X_test, y, cat_features = preprocess(train, test)
print(f'X shape     : {X.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'카테고리 피처 수: {len(cat_features)}')

## 4. Train / Validation 분리 (8:2)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y          # 클래스 비율 유지
)

print(f'X_train : {X_train.shape}  |  y_train 분포: {dict(y_train.value_counts())}')
print(f'X_val   : {X_val.shape}  |  y_val   분포: {dict(y_val.value_counts())}')

## 5. 베이스 모델 학습

In [ ]:
# ── 공통 평가 함수 ──────────────────────────────────────────
def evaluate(name, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)
    pred   = model.predict(X_v)
    prob   = model.predict_proba(X_v)[:, 1] if hasattr(model, 'predict_proba') else pred
    acc    = accuracy_score(y_v, pred)
    f1     = f1_score(y_v, pred, average='binary')
    auc    = roc_auc_score(y_v, prob)
    print(f'[{name:25s}]  ACC={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    return {'name': name, 'model': model, 'acc': acc, 'f1': f1, 'auc': auc}

results = []

In [ ]:
# ── LightGBM ────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    verbose=-1
)
results.append(evaluate('LightGBM', lgb_model, X_train, y_train, X_val, y_val))

In [ ]:
# ── XGBoost ─────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0
)
results.append(evaluate('XGBoost', xgb_model, X_train, y_train, X_val, y_val))

In [ ]:
# ── CatBoost ────────────────────────────────────────────────
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=SEED,
    verbose=0
)
results.append(evaluate('CatBoost', cat_model, X_train, y_train, X_val, y_val))

In [ ]:
# ── Random Forest ───────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1
)
results.append(evaluate('RandomForest', rf_model, X_train, y_train, X_val, y_val))

In [ ]:
# ── Logistic Regression (메타 모델용으로도 활용) ─────────────
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED))
])
results.append(evaluate('LogisticRegression', lr_model, X_train, y_train, X_val, y_val))

## 6. 앙상블

### 6-1. Soft Voting

In [ ]:
voting = VotingClassifier(
    estimators=[
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('cat', cat_model),
        ('rf',  rf_model),
    ],
    voting='soft'
)
results.append(evaluate('SoftVoting', voting, X_train, y_train, X_val, y_val))

### 6-2. Weighted Average (확률 평균)

In [ ]:
# 각 모델 AUC 기반 가중치 산출
base_models = [
    ('LightGBM', lgb_model),
    ('XGBoost',  xgb_model),
    ('CatBoost', cat_model),
    ('RandomForest', rf_model),
]

val_probs = np.column_stack([
    m.predict_proba(X_val)[:, 1] for _, m in base_models
])

# 각 모델 AUC
aucs = [roc_auc_score(y_val, val_probs[:, i]) for i in range(len(base_models))]
weights = np.array(aucs) / sum(aucs)
print('모델별 AUC 가중치:')
for (name, _), w, a in zip(base_models, weights, aucs):
    print(f'  {name:20s}: AUC={a:.4f}  weight={w:.4f}')

weighted_prob = val_probs @ weights
weighted_pred = (weighted_prob >= 0.5).astype(int)

acc = accuracy_score(y_val, weighted_pred)
f1  = f1_score(y_val, weighted_pred)
auc = roc_auc_score(y_val, weighted_prob)
print(f'\n[{"WeightedAverage":25s}]  ACC={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
results.append({'name': 'WeightedAverage', 'model': None, 'acc': acc, 'f1': f1, 'auc': auc})

### 6-3. Stacking

In [ ]:
stacking = StackingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=-1)),
        ('xgb', xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, use_label_encoder=False,
                                   eval_metric='logloss', random_state=SEED, verbosity=0)),
        ('rf',  RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=5,
    passthrough=False
)
results.append(evaluate('Stacking', stacking, X_train, y_train, X_val, y_val))

## 7. 모델 비교 & 최적 모델 선택

In [ ]:
results_df = pd.DataFrame(results)[['name','acc','f1','auc']].sort_values('auc', ascending=False)
display(results_df.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.25
ax.bar(x - w, results_df['acc'], w, label='ACC',  color='#5B9BD5')
ax.bar(x,     results_df['f1'],  w, label='F1',   color='#ED7D31')
ax.bar(x + w, results_df['auc'], w, label='AUC',  color='#70AD47')
ax.set_xticks(x)
ax.set_xticklabels(results_df['name'], rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title('모델 성능 비교 (Validation Set)')
plt.tight_layout()
plt.show()

In [ ]:
# 최적 모델 혼동 행렬
best_name = results_df.iloc[0]['name']
best_result = next(r for r in results if r['name'] == best_name)
best_model = best_result['model']
print(f'최적 모델: {best_name}')

if best_model is not None:
    pred_best = best_model.predict(X_val)
    print(classification_report(y_val, pred_best))
    cm = confusion_matrix(y_val, pred_best)
    ConfusionMatrixDisplay(cm).plot()
    plt.title(f'Confusion Matrix - {best_name}')
    plt.show()

## 8. 피처 중요도

In [ ]:
# LightGBM 피처 중요도
fi = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
fi = fi.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
fi.sort_values().plot(kind='barh', color='steelblue')
plt.title('LightGBM Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

## 9. 최종 예측 & 제출 파일 생성

In [ ]:
# ── 전체 학습 데이터로 재학습 후 test 예측 ─────────────────
# (Weighted Average 기준, 필요에 따라 best_model.predict 로 교체)

final_models = [
    lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1),
    xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, use_label_encoder=False,
                      eval_metric='logloss', random_state=SEED, verbosity=0),
    CatBoostClassifier(iterations=500, learning_rate=0.05, random_seed=SEED, verbose=0),
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
]

test_probs = []
for m in final_models:
    m.fit(X, y)  # 전체 학습 데이터 사용
    test_probs.append(m.predict_proba(X_test)[:, 1])

# AUC 기반 가중치 재사용
final_prob = np.column_stack(test_probs) @ weights
final_pred = (final_prob >= 0.5).astype(int)

sub['completed'] = final_pred
sub.to_csv(f'{DATA_PATH}/submission.csv', index=False)
print('제출 파일 저장 완료:')
print(sub['completed'].value_counts())
sub.head()